[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iitm-da/da2402/blob/master/intro/practice/pandas_worksheet.ipynb)

# Practice · pandas

DA2402 · Data Curation and Visualization · Dr. Arun B Ayyar

Nine questions on one dataset. Each question names a variable. Put your result in that variable and
run the cell. The worked answer sits under **Answer**. Click it open once you have tried.

**The data.** Six months of daily entry and exit counts for 18 Chennai Metro stations, written for
this worksheet. Real station names, invented counts.

- `metro_daily.csv`: `date`, `station`, `entries`, `exits`, `fare_collected` (rupees)
- `metro_stations.csv`: `station`, `line`, `zone`, `opened`, `interchange`


In [ ]:
import numpy as np
import pandas as pd

URL = "https://raw.githubusercontent.com/iitm-da/da2402/master/intro/practice/data/"
daily = pd.read_csv(URL + "metro_daily.csv", parse_dates=["date"])
stations = pd.read_csv(URL + "metro_stations.csv")

daily.head()

### Q1  Size of the file

How many rows and columns are in `daily`?

Answer variable `q1`: a tuple `(rows, columns)`.

In [ ]:
q1 = ...   # your answer
q1

<details>
<summary><b>Answer</b></summary>

```python
q1 = daily.shape
q1
```

```
(3258, 5)
```

</details>

### Q2  Busy station-days

Count the rows where `entries` is above 20,000. One row is one station on one day.

Answer variable `q2`: an `int`.

In [ ]:
q2 = ...   # your answer
q2

<details>
<summary><b>Answer</b></summary>

```python
q2 = int((daily["entries"] > 20000).sum())
q2
```

```
159
```

</details>

### Q3  Net flow

Add a column `net`, entries minus exits. Report its mean over the whole file, rounded
to 2 decimals.

Answer variable `q3`: a `float`.

In [ ]:
q3 = ...   # your answer
q3

<details>
<summary><b>Answer</b></summary>

```python
daily["net"] = daily["entries"] - daily["exits"]
q3 = float(daily["net"].mean().round(2))
q3
```

```
-17.07
```

</details>

### Q4  Top five stations

Total `entries` per station over the six months. Keep the five largest, in descending
order.

Answer variable `q4`: a Series indexed by station.

In [ ]:
q4 = ...   # your answer
q4

<details>
<summary><b>Answer</b></summary>

```python
q4 = daily.groupby("station")["entries"].sum().sort_values(ascending=False).head(5)
q4
```

```
station
Chennai Central    3878155
Alandur            2975714
CMBT               2724415
Egmore             2462107
Guindy             2253643
Name: entries, dtype: int64
```

</details>

### Q5  Fare by line

`daily` has no line column. Bring it in from `stations`, then total `fare_collected` per
line. Report it in rupees million, rounded to 2 decimals.

Answer variable `q5`: a Series indexed by line.

In [ ]:
q5 = ...   # your answer
q5

<details>
<summary><b>Answer</b></summary>

```python
merged = daily.merge(stations, on="station")
q5 = (merged.groupby("line")["fare_collected"].sum() / 1e6).round(2)
q5
```

```
line
Blue     529.99
Green    447.27
Name: fare_collected, dtype: float64
```

</details>

### Q6  Month against line

Total `entries` with month down the rows and line across the columns.

Answer variable `q6`: a DataFrame, 6 rows by 2 columns.

In [ ]:
q6 = ...   # your answer
q6

<details>
<summary><b>Answer</b></summary>

```python
merged["month"] = merged["date"].dt.month
q6 = merged.pivot_table(index="month", columns="line", values="entries", aggfunc="sum")
q6
```

```
line      Blue    Green
month                  
1      2713437  2249636
2      2573761  2126102
3      3065335  2560637
4      2883446  2387161
5      2894025  2344404
6      2689197  2210113
```

</details>

### Q7  Monthly totals by resampling

The same monthly totals as Q6, for the whole network, computed on a date index with
`resample` instead of a `month` column.

Answer variable `q7`: a Series indexed by month end.

In [ ]:
q7 = ...   # your answer
q7

<details>
<summary><b>Answer</b></summary>

```python
q7 = daily.set_index("date")["entries"].resample("ME").sum()
q7
```

```
date
2025-01-31    4963073
2025-02-28    4699863
2025-03-31    5625972
2025-04-30    5270607
2025-05-31    5238429
2025-06-30    4899310
Freq: ME, Name: entries, dtype: int64
```

</details>

### Q8  Busiest station on each line

For each line, the one station with the highest six-month `entries` total.

Answer variable `q8`: a Series indexed by line, holding station names.

In [ ]:
q8 = ...   # your answer
q8

<details>
<summary><b>Answer</b></summary>

```python
totals = merged.groupby(["line", "station"])["entries"].sum().reset_index()
q8 = totals.loc[totals.groupby("line")["entries"].idxmax()].set_index("line")["station"]
q8
```

```
line
Blue     Chennai Central
Green               CMBT
Name: station, dtype: object
```

</details>

### Q9  Weekday against weekend

Mean `entries` per station-day, split by line and by whether the date falls on a Saturday
or Sunday. Round to whole riders.

Answer variable `q9`: a DataFrame, 2 rows by 2 columns.

In [ ]:
q9 = ...   # your answer
q9

<details>
<summary><b>Answer</b></summary>

```python
merged["day_type"] = np.where(merged["date"].dt.dayofweek >= 5, "weekend", "weekday")
q9 = merged.pivot_table(index="line", columns="day_type", values="entries", aggfunc="mean").round(0)
q9
```

```
day_type  weekday  weekend
line                      
Blue      11587.0   7193.0
Green      9558.0   5943.0
```

</details>